# 量化交易入门 Vol.4：回测基础与绩效分析

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/greathousesh/qlora-sft-tutorial/blob/main/quant/04_backtesting.ipynb)

> **Kaggle 一键运行**：点击上方按钮 → 选择 **"CPU"** → Run All

## 本节你将学到

| 知识点 | 说明 |
|--------|------|
| **回测的定义与局限** | 回测能告诉你什么，不能告诉你什么 |
| **多空组合构建** | 从预测信号到可交易的组合 |
| **核心绩效指标** | Sharpe, MaxDD, Calmar, IR, Turnover |
| **交易成本影响** | 费率对策略收益的侵蚀有多严重 |
| **常见陷阱** | 前视偏差、幸存者偏差、过拟合 |
| **绩效归因** | 收益来自哪里？ |

## 课程位置

本节将使用 Vol.3 训练的 ML 模型预测信号进行回测。  
如果没有运行 Vol.3，本节也会重新生成一个简单的动量信号作为替代。

In [ ]:
import subprocess, sys
pkgs = ["yfinance>=0.2.30", "pandas>=1.5.0", "numpy>=1.23.0",
        "matplotlib>=3.6.0", "seaborn>=0.12.0", "scipy>=1.9.0"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("✅ 依赖安装完毕")

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import spearmanr

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11,
                     'axes.titlesize': 12, 'axes.grid': True, 'grid.alpha': 0.3})

TICKERS = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'TSLA',
           'JPM', 'GS', 'JNJ', 'WMT', 'XOM', 'CVX', 'PG', 'KO', 'DIS']
BENCHMARK = 'SPY'
START, END = '2019-01-01', '2024-01-01'

raw      = yf.download(TICKERS + [BENCHMARK], start=START, end=END, auto_adjust=True, progress=False)
prices   = raw['Close'].dropna(how='all')
log_rets = np.log(prices / prices.shift(1))

# 尝试加载 Vol.3 的预测结果，没有则用动量信号替代
if os.path.exists('ml_predictions.csv'):
    preds_df = pd.read_csv('ml_predictions.csv', parse_dates=['date'])
    print("✅ 加载了 Vol.3 的 ML 预测信号")
else:
    print("ℹ️  未找到 ml_predictions.csv，使用动量因子作为信号替代")
    # 构建简单动量信号
    momentum = prices[TICKERS].pct_change(63).shift(1)  # 3个月动量，shift避免未来泄漏
    rows = []
    for date in momentum['2022-01-01':'2023-12-31'].index:
        for ticker in TICKERS:
            v = momentum.loc[date, ticker]
            if not np.isnan(v):
                rows.append({'date': date, 'ticker': ticker, 'pred': v})
    preds_df = pd.DataFrame(rows)

print(f"预测信号: {len(preds_df)} 行, 覆盖 {preds_df['date'].nunique()} 天")

## Step 1：回测是什么？能相信多少？

### 回测的本质

**回测（Backtesting）** = 假设过去已经按照策略操作，看看会赚多少钱。

它是策略研究的必要工具，但有很重要的局限性：

```
回测能告诉你：
  ✅ 策略的历史逻辑是否自洽
  ✅ 大致的风险收益特征
  ✅ 在不同市场环境下的表现

回测不能告诉你：
  ❌ 未来一定能复制历史
  ❌ 市场冲击成本的真实影响
  ❌ 策略规模化后是否仍有效
  ❌ 你是否只是过拟合了历史
```

### 回测的基本假设

一个「干净」的回测需要满足：
1. **无未来数据泄漏**：每天的信号只能用到当天收盘前的数据
2. **真实成交假设**：考虑买卖价差、市场冲击、交易成本
3. **换仓假设**：什么价格成交？（通常假设次日开盘价）
4. **仓位约束**：是否允许做空？是否有杠杆限制？

## Step 2：多空组合构建

**量化选股的经典策略：等权多空**

```
每个换仓日（如每周或每月）：

1. 根据模型信号，对所有股票排名

2. 做多（Long）：信号最高的 Top-K 只股票
   各股等权分配

3. 做空（Short）：信号最低的 Bottom-K 只股票
   各股等权分配

4. 多空对冲：Long 组合收益 - Short 组合收益
   （理论上 Beta 中性，主要暴露 Alpha）
```

**纯多头策略（Long-Only）**：只做多，适合不允许做空的账户。

In [ ]:
REBALANCE_FREQ = 'W'  # 每周换仓（W=周, M=月, 2W=双周）
N_LONG  = 3           # 做多前 N 只
N_SHORT = 3           # 做空后 N 只（做空需要借券，简化假设可以做空）

def build_portfolio(preds_df, prices_df, rebalance_freq='W',
                    n_long=3, n_short=3, cost_bps=10):
    """
    从预测信号构建多空组合。
    
    Parameters
    ----------
    cost_bps : float  交易成本，单位 basis points（1 bps = 0.01%）
    """
    # 确定换仓日（每个 rebalance_freq 的第一个交易日）
    all_dates = sorted(preds_df['date'].unique())
    date_series = pd.Series(all_dates, index=all_dates)
    rebal_dates = set(date_series.resample(rebalance_freq).first().dropna())
    
    log_ret = np.log(prices_df / prices_df.shift(1))
    
    portfolio_returns = []
    long_holdings  = {}  # ticker -> weight
    short_holdings = {}
    prev_long_tickers  = set()
    prev_short_tickers = set()
    
    for date in sorted(log_ret.index):
        if str(date)[:10] == str(all_dates[0])[:10]:
            continue
        
        # 换仓日：更新持仓
        if date in rebal_dates:
            day_preds = preds_df[preds_df['date'] == date]
            if len(day_preds) < n_long + n_short:
                pass  # 样本不够，保持现有持仓
            else:
                sorted_preds = day_preds.sort_values('pred', ascending=False)
                
                new_long  = set(sorted_preds.iloc[:n_long]['ticker'].tolist())
                new_short = set(sorted_preds.iloc[-n_short:]['ticker'].tolist())
                
                # 计算换手率
                turnover_l = len(new_long.symmetric_difference(prev_long_tickers)) / (2 * n_long + 1e-10)
                turnover_s = len(new_short.symmetric_difference(prev_short_tickers)) / (2 * n_short + 1e-10)
                cost_today = (turnover_l + turnover_s) / 2 * cost_bps * 1e-4
                
                long_holdings  = {t: 1/n_long  for t in new_long}
                short_holdings = {t: -1/n_short for t in new_short}
                prev_long_tickers  = new_long
                prev_short_tickers = new_short
            else:
                cost_today = 0
        else:
            cost_today = 0
        
        # 当日收益
        if date not in log_ret.index:
            continue
        day_ret = log_ret.loc[date]
        
        long_ret  = sum(w * day_ret.get(t, 0) for t, w in long_holdings.items())
        short_ret = sum(w * day_ret.get(t, 0) for t, w in short_holdings.items())
        net_ret   = (long_ret + short_ret) - cost_today  # 扣除交易成本
        
        portfolio_returns.append({
            'date':       date,
            'long_ret':   long_ret,
            'short_ret':  short_ret,
            'net_ret':    net_ret,
            'n_long':     len(long_holdings),
            'n_short':    len(short_holdings),
        })
    
    return pd.DataFrame(portfolio_returns).set_index('date')

# 构建多空组合（考虑 10 bps 交易成本）
port_df = build_portfolio(preds_df, prices[TICKERS], REBALANCE_FREQ,
                          N_LONG, N_SHORT, cost_bps=10)

print(f"组合构建完成: {len(port_df)} 个交易日")
print(f"平均每天持仓: 多头 {port_df['n_long'].mean():.1f} 只, 空头 {port_df['n_short'].mean():.1f} 只")

## Step 3：核心绩效指标

| 指标 | 公式 | 含义 | 参考值 |
|------|------|------|--------|
| **年化收益** | $(1+r)^{252/N} - 1$ | 换算成每年赚多少 | >10% 较好 |
| **年化波动** | $\sigma \times \sqrt{252}$ | 策略的风险 | 越小越好 |
| **Sharpe Ratio** | $\mu / \sigma$ | 每单位风险的收益 | >1.0 良好 |
| **最大回撤** | $\min(\frac{V_t - V_{peak}}{V_{peak}})$ | 最惨时亏多少 | <15% 较好 |
| **Calmar Ratio** | 年化收益 / 最大回撤 | 用损失衡量收益 | >1.0 良好 |
| **换手率** | 每次换仓的仓位变化比例 | 越高交易成本越高 | 取决于策略 |

In [ ]:
def performance_report(returns_series, label='Strategy', annual_factor=252):
    """计算完整绩效指标"""
    r = returns_series.dropna()
    cum = (1 + r).cumprod()
    
    # 年化收益
    total_ret = cum.iloc[-1] - 1
    n_years   = len(r) / annual_factor
    ann_ret   = (1 + total_ret) ** (1 / n_years) - 1
    
    # 年化波动
    ann_vol = r.std() * np.sqrt(annual_factor)
    
    # Sharpe
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    
    # 最大回撤
    peak    = cum.cummax()
    dd      = (cum - peak) / peak
    max_dd  = dd.min()
    max_dd_end   = dd.idxmin()
    max_dd_start = cum[:max_dd_end].idxmax()
    
    # Calmar
    calmar = ann_ret / abs(max_dd) if max_dd != 0 else 0
    
    # 胜率
    win_rate = (r > 0).mean()
    
    # 盈亏比（平均盈利 / 平均亏损绝对值）
    avg_win  = r[r > 0].mean() if (r > 0).any() else 0
    avg_loss = abs(r[r < 0].mean()) if (r < 0).any() else 1e-10
    profit_loss_ratio = avg_win / avg_loss
    
    return {
        'label':          label,
        '年化收益 %':     ann_ret * 100,
        '年化波动 %':     ann_vol * 100,
        'Sharpe Ratio':   sharpe,
        '最大回撤 %':     max_dd * 100,
        'Calmar Ratio':   calmar,
        '日胜率 %':       win_rate * 100,
        '盈亏比':         profit_loss_ratio,
        '总收益 %':       total_ret * 100,
        'MDD 开始':       str(max_dd_start.date()),
        'MDD 结束':       str(max_dd_end.date()),
    }

# 基准：SPY
spy_ret = log_rets[BENCHMARK][port_df.index[0]:port_df.index[-1]]
spy_report   = performance_report(spy_ret, 'SPY 基准')
port_report  = performance_report(port_df['net_ret'], '多空组合')
long_report  = performance_report(port_df['long_ret'], '纯多头')

print("\n绩效对比报告")
print("=" * 60)
for key in ['年化收益 %', '年化波动 %', 'Sharpe Ratio', '最大回撤 %',
            'Calmar Ratio', '日胜率 %', '盈亏比', '总收益 %']:
    print(f"  {key:<15}  {spy_report[key]:>10.2f}  {long_report[key]:>10.2f}  {port_report[key]:>10.2f}")
print()
for rpt in [spy_report, long_report, port_report]:
    print(f"  {rpt['label']} 最大回撤期: {rpt['MDD 开始']} → {rpt['MDD 结束']}")

In [ ]:
# 完整绩效可视化
fig = plt.figure(figsize=(16, 12))
gs  = fig.add_gridspec(3, 2, hspace=0.4, wspace=0.3)

# 图1：累计净值曲线
ax1 = fig.add_subplot(gs[0, :])

cum_port = (1 + port_df['net_ret']).cumprod()
cum_long = (1 + port_df['long_ret']).cumprod()
cum_spy  = (1 + spy_ret).cumprod()
cum_spy  = cum_spy.reindex(port_df.index).ffill()

ax1.plot(cum_port.index, cum_port, 'steelblue', lw=2.5, label='多空组合（含成本）')
ax1.plot(cum_long.index, cum_long, 'green',     lw=2,   ls='--', label='纯多头', alpha=0.8)
ax1.plot(cum_spy.index,  cum_spy,  'gray',      lw=1.5, ls='--', label='SPY 基准', alpha=0.7)
ax1.fill_between(cum_port.index, 1, cum_port,
                 where=cum_port >= 1, alpha=0.1, color='steelblue')
ax1.fill_between(cum_port.index, 1, cum_port,
                 where=cum_port < 1, alpha=0.15, color='red')
ax1.axhline(1, color='black', lw=0.8, ls=':')
ax1.set_title('策略净值曲线（从 1 开始）')
ax1.set_ylabel('Cumulative Return')
ax1.legend()
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# 图2：回撤
ax2 = fig.add_subplot(gs[1, 0])
peak = cum_port.cummax()
dd   = (cum_port - peak) / peak
ax2.fill_between(dd.index, dd, 0, alpha=0.6, color='red')
ax2.plot(dd.index, dd, color='darkred', lw=1)
ax2.set_title(f'回撤曲线  (最大回撤: {dd.min()*100:.1f}%)')
ax2.set_ylabel('Drawdown')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x*100:.0f}%'))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# 图3：滚动年化 Sharpe
ax3 = fig.add_subplot(gs[1, 1])
rolling_sharpe = port_df['net_ret'].rolling(60).apply(
    lambda x: x.mean() / x.std() * np.sqrt(252) if x.std() > 0 else 0
)
ax3.plot(rolling_sharpe.index, rolling_sharpe, color='purple', lw=1.5)
ax3.axhline(0,   color='black', lw=1)
ax3.axhline(1.0, color='green', lw=1, ls='--', label='Sharpe=1.0')
ax3.axhline(-1.0, color='red',  lw=1, ls='--', label='Sharpe=-1.0')
ax3.fill_between(rolling_sharpe.index, 0, rolling_sharpe,
                 where=rolling_sharpe >= 0, alpha=0.2, color='green')
ax3.fill_between(rolling_sharpe.index, 0, rolling_sharpe,
                 where=rolling_sharpe < 0,  alpha=0.2, color='red')
ax3.set_title('60日滚动 Sharpe Ratio')
ax3.set_ylabel('Sharpe')
ax3.legend()
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# 图4：月度收益热力图
ax4 = fig.add_subplot(gs[2, :])
monthly_ret = port_df['net_ret'].resample('M').apply(lambda x: (1+x).prod()-1)
monthly_pivot = monthly_ret.groupby([monthly_ret.index.year, monthly_ret.index.month]).mean().unstack()
monthly_pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                          'Jul','Aug','Sep','Oct','Nov','Dec'][:len(monthly_pivot.columns)]

sns.heatmap(monthly_pivot * 100, ax=ax4,
            annot=True, fmt='.1f', cmap='RdYlGn',
            center=0, vmin=-5, vmax=5,
            linewidths=0.5, cbar_kws={'label': 'Monthly Return %'})
ax4.set_title('月度收益热力图（单位 %）')
ax4.set_xlabel('')
ax4.set_ylabel('Year')

plt.suptitle('量化策略完整绩效分析', fontsize=14, y=1.01)
plt.savefig('backtest_results.png', bbox_inches='tight')
plt.show()

## Step 4：交易成本的影响 —— 被忽视的利润杀手

很多初学者在回测中忽略交易成本，导致真实交易时亏钱。  
**交易成本来源**：
1. **佣金（Commission）**：0.5-2 bps 每笔（美股在线经纪商约 0）
2. **买卖价差（Bid-Ask Spread）**：通常 1-5 bps，流动性差的股票更高
3. **市场冲击（Market Impact）**：大单拉动价格，约 5-20 bps
4. **滑点（Slippage）**：实际成交价 vs 预期价，约 1-5 bps

**合理假设**：散户约 10-20 bps，机构约 5-10 bps（单边，买和卖各收一次）

In [ ]:
# 不同交易成本下的策略表现
cost_scenarios = [0, 5, 10, 20, 50]  # 单位 bps

results = []
for cost_bps in cost_scenarios:
    port = build_portfolio(preds_df, prices[TICKERS], REBALANCE_FREQ,
                           N_LONG, N_SHORT, cost_bps=cost_bps)
    perf = performance_report(port['net_ret'], f'{cost_bps}bps')
    perf['cost_bps'] = cost_bps
    results.append(perf)

cost_df = pd.DataFrame(results).set_index('cost_bps')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric in zip(axes, ['年化收益 %', 'Sharpe Ratio', '最大回撤 %']):
    values = cost_df[metric]
    bars = ax.bar(cost_df.index.astype(str) + 'bps', values,
                  color=['green' if v > 0 else 'red' for v in values],
                  alpha=0.8, edgecolor='white')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2.,
                bar.get_height() + (0.1 if val >= 0 else -0.5),
                f'{val:.2f}', ha='center', va='bottom', fontsize=10)
    ax.axhline(0, color='black', lw=1)
    ax.set_xlabel('单边交易成本 (basis points)')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} vs 交易成本')

plt.suptitle('交易成本敏感性分析\n成本越高，策略收益越低 —— 找到盈亏平衡点', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('cost_sensitivity.png', bbox_inches='tight')
plt.show()

print("\n不同成本下的绩效（年化收益 % / Sharpe）:")
for _, row in cost_df.iterrows():
    viable = "✅" if row['Sharpe Ratio'] > 0.5 else "⚠️ "
    print(f"  {viable} {row['label']:>8}: 年化={row['年化收益 %']:>6.2f}%, Sharpe={row['Sharpe Ratio']:>5.2f}")

## Step 5：回测常见陷阱详解

### 陷阱 1：前视偏差（Look-Ahead Bias）

```python
# ❌ 错误：用当天收盘价计算信号，再用当天收盘价成交
# 实际上下午3点才能知道收盘价，不可能用它来当天交易
signal = prices.pct_change(21)        # 今天收盘价计算的动量
order  = signal                         # 当天收盘成交 → 前视！

# ✅ 正确：昨天的信号，今天的价格成交
signal = prices.pct_change(21).shift(1)  # 昨天计算的信号
order  = signal                           # 今天成交 → 正确
```

### 陷阱 2：幸存者偏差（Survivorship Bias）

```
我们只下载了「今天还存在」的股票历史数据。
那些曾经存在但后来退市、破产的公司被我们忽略了。
→ 历史数据里幸存下来的都是「好公司」
→ 基于此的回测结果会高估策略的真实表现

解决方案：使用含历史退市股票的完整数据库（如 Compustat, Bloomberg）
```

### 陷阱 3：换手率太高

每次换仓都要付交易成本。如果一个策略每天换仓，10bps × 252 = 25.2% 的年化成本！

In [ ]:
# 换仓频率 vs 策略表现
freq_scenarios = [
    ('D', '每天'),   # 危险！成本极高
    ('W', '每周'),
    ('2W', '双周'),
    ('M', '每月'),
]

print("换仓频率敏感性分析（成本固定 10bps）：")
print(f"{'频率':<8} {'年化换手%':>12} {'年化收益%':>12} {'Sharpe':>10}")
print("-" * 48)

freq_results = []
for freq, label in freq_scenarios:
    port = build_portfolio(preds_df, prices[TICKERS], freq, N_LONG, N_SHORT, cost_bps=10)
    perf = performance_report(port['net_ret'], label)
    
    # 估算年化换手率
    periods_per_year = {'D': 252, 'W': 52, '2W': 26, 'M': 12}.get(freq, 52)
    est_turnover = periods_per_year * 0.5 * 100  # 每次约 50% 换手
    
    viable = "✅" if perf['Sharpe Ratio'] > 0.3 else "⚠️ "
    print(f"  {viable} {label:<6} {est_turnover:>12.0f}% {perf['年化收益 %']:>12.2f}% {perf['Sharpe Ratio']:>10.3f}")
    freq_results.append({'freq': label, **perf})

print("\n建议：从月度或双周换仓开始，交易成本可控")
print("每日换仓适合大资金+极低成本（机构可用），散户通常不划算")

## Step 6：收益归因 —— 钱到底是哪里赚的？

量化策略的收益可以分解为：
1. **市场 Beta 收益**：跟着大盘涨跌（不算本事）
2. **Alpha 收益**：超出大盘的超额收益（才是策略真正的价值）
3. **因子暴露收益**：对某个风险因子（如价值/成长）的暴露

In [ ]:
from numpy.linalg import lstsq

# 将策略收益对 SPY 做线性回归，分解 Alpha 和 Beta
port_ret  = port_df['net_ret']
spy_ret_aligned = spy_ret.reindex(port_ret.index).dropna()
port_ret_aligned = port_ret.reindex(spy_ret_aligned.index).dropna()

X = np.column_stack([np.ones(len(spy_ret_aligned)), spy_ret_aligned])
y = port_ret_aligned.values
beta_hat, _, _, _ = lstsq(X, y, rcond=None)

alpha_daily = beta_hat[0]   # 日 alpha
beta        = beta_hat[1]   # 市场 beta

alpha_annual = alpha_daily * 252
corr_spy = np.corrcoef(spy_ret_aligned, port_ret_aligned)[0, 1]

print("收益归因分析：")
print(f"  市场 Beta:     {beta:.4f}  (1.0=随大盘，0=市场中性)")
print(f"  日 Alpha:      {alpha_daily*100:.4f}%")
print(f"  年化 Alpha:    {alpha_annual*100:.2f}%")
print(f"  与SPY相关性:   {corr_spy:.3f}")
print(f"\n解读：")
if abs(beta) < 0.3:
    print(f"  策略 Beta={beta:.2f}，接近 0 → 市场中性策略，不依赖大盘方向")
elif beta > 0.3:
    print(f"  策略 Beta={beta:.2f}，有一定多头偏向，大盘跌时会有损失")

if alpha_annual > 0.05:
    print(f"  年化 Alpha={alpha_annual*100:.1f}%，策略有正 Alpha（真实选股能力）")
else:
    print(f"  年化 Alpha={alpha_annual*100:.1f}%，Alpha 较低，需要优化策略")

# 可视化：策略收益 vs SPY 收益散点图
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(spy_ret_aligned * 100, port_ret_aligned * 100,
           alpha=0.3, s=5, color='steelblue')
x_line = np.linspace(spy_ret_aligned.min(), spy_ret_aligned.max(), 100)
y_line = beta_hat[0] + beta_hat[1] * x_line
ax.plot(x_line * 100, y_line * 100, 'r-', lw=2,
        label=f'拟合: α={alpha_daily*100:.4f}%, β={beta:.3f}')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('SPY 日收益率 %')
ax.set_ylabel('策略日收益率 %')
ax.set_title('策略 vs 市场散点图\n(Alpha = 截距, Beta = 斜率)')
ax.legend()

ax2 = axes[1]
port_cum = (1 + port_ret_aligned).cumprod()
spy_cum  = (1 + spy_ret_aligned).cumprod()
alpha_cum = port_cum / spy_cum  # 相对净值
ax2.plot(alpha_cum.index, alpha_cum, color='steelblue', lw=2)
ax2.axhline(1, color='black', lw=1, ls='--')
ax2.fill_between(alpha_cum.index, 1, alpha_cum,
                 where=alpha_cum > 1, alpha=0.2, color='green')
ax2.fill_between(alpha_cum.index, 1, alpha_cum,
                 where=alpha_cum < 1, alpha=0.2, color='red')
ax2.set_title('相对净值 = 策略 / SPY\n>1 表示跑赢基准')
ax2.set_ylabel('Relative NAV')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('attribution.png', bbox_inches='tight')
plt.show()

## 本节总结

```
回测完整流程

预测信号（Vol.3 ML 模型）
    │
    ├── 组合构建
    │       ├── 按信号排名
    │       ├── 多头：Top-N 等权
    │       └── 空头：Bottom-N 等权
    │
    ├── 回测执行（考虑交易成本）
    │       ├── 换仓频率
    │       └── 成本 = 换手率 × 单位成本
    │
    ├── 绩效评估
    │       ├── 年化收益、Sharpe、最大回撤、Calmar
    │       └── 月度收益热力图
    │
    ├── 敏感性分析
    │       ├── 成本敏感性（关键！）
    │       └── 换仓频率敏感性
    │
    └── 收益归因（Alpha vs Beta）
```

### 一个好的量化策略需要同时满足：

| 条件 | 原因 |
|------|------|
| Sharpe > 1.0 | 收益/风险比值得交易 |
| 扣费后正收益 | 考虑实际成本后仍可盈利 |
| 最大回撤可接受 | 你真的能扛住最惨时期 |
| 样本外有效 | 不只是历史拟合好 |
| 容量足够 | 资金规模内仍有效 |

## 下一步：Vol.5 qlib 完整工作流

前 4 节我们用 pandas + yfinance 从头实现了整个量化研究流程。  
最后一节将展示 **qlib 如何把这一切系统化、工程化**，提升研究效率。

---
*Vol.4 完 | 课程：量化交易从入门到 qlib*